# Laboratorio 3: Generacion de Datos Sinteticos y Modelado de Ruido Observacional

## 1. Introduccion: El Laboratorio in Silico

En la practica de la ciencia de datos aplicada a sistemas dinamicos, rara vez se dispone de trayectorias continuas exactas y libres de error. Los registros clinicos y epidemiologicos se recolectan a intervalos discretos de tiempo (por ejemplo, reportes hospitalarios diarios) y estan afectados por diversas fuentes de incertidumbre: errores de muestreo, subnotificacion, sensibilidad imperfecta de las pruebas diagnosticas y demoras en el procesamiento de datos.

Para desarrollar y validar algoritmos de calibracion e inferencia de parametros (problemas inversos), es necesario contar con un entorno de prueba controlado: un **laboratorio in silico**. En este laboratorio generamos observaciones sinteticas a partir de un modelo mecanistico cuya **verdad de campo** (*Ground Truth*) conocemos con precision absoluta. Esto permite cuantificar de forma rigurosa la exactitud, robustez y sesgo de los metodos de estimacion.

## Objetivos de Aprendizaje

1. Simular la verdad de campo continua del modelo SIR ($\beta = 0.4$, $\gamma = 0.1$) mediante RK45 adaptativo.
2. Implementar el submuestreo temporal discreto en dias enteros $t_k \in \{0, 1, 2, \dots, 50\}$.
3. Modelar el proceso observacional incorporando ruido Gaussiano $\mathcal{N}(0, \sigma^2)$ con proyeccion a no-negatividad fisica: $I^{\text{obs}} = \max(0, I + \varepsilon)$.
4. Efectuar un analisis estadistico riguroso de los residuos observacionales (media, desviacion estandar, banda $\pm 2\sigma$).
5. Visualizar las observaciones clinicas y caracterizar la incertidumbre instrumental.


In [ ]:
# Celda de configuracion y librerias cientificas
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
from typing import Any
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "figure.titlesize": 13,
    "lines.linewidth": 2.0,
    "grid.alpha": 0.4
})

print("Configuracion inicial completada.")


## 2. Generacion de la Verdad de Campo (Ground Truth)

Consideramos el sistema continuo SIR:

$$\frac{dS}{dt} = -\beta S I$$
$$\frac{dI}{dt} = \beta S I - \gamma I$$
$$\frac{dR}{dt} = \gamma I$$

con parametros nominales verdaderos:
- $\beta^* = 0.40$ (tasa de transmision)
- $\gamma^* = 0.10$ (tasa de recuperacion)
- $R_0^* = \beta^* / \gamma^* = 4.00$
- Condiciones iniciales: $S(0) = 0.99$, $I(0) = 0.01$, $R(0) = 0.00$
- Horizonte temporal: $T = 50.0$ dias

Generamos la trayectoria continua evaluada en una malla densa con RK45 adaptativo.


In [ ]:
def generar_verdad_campo(
    beta: float = 0.4,
    gamma: float = 0.1,
    S0: float = 0.99,
    I0: float = 0.01,
    T: float = 50.0,
    dt_eval: float = 0.005
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Genera la solucion continua de referencia (Ground Truth) para el modelo SIR.
    """
    # TU CODIGO AQUI
    ...


In [ ]:
# Pruebas de validacion para generar_verdad_campo
t_ref, S_ref, I_ref, R_ref = generar_verdad_campo(0.4, 0.1, 0.99, 0.01, 50.0)
assert np.isclose(S_ref[0], 0.99) and np.isclose(I_ref[0], 0.01), "Condiciones iniciales incorrectas."
assert np.max(np.abs(S_ref + I_ref + R_ref - 1.0)) < 1e-6, "Conservacion de masa no satisfecha."
print("Validacion de generar_verdad_campo: EXITOSA.")


## 3. Muestreo en Tiempos de Reporte Clinico Discreto

En los sistemas de vigilancia epidemiologica, las notificaciones se registran en dias enteros: $t_k \in \{0, 1, 2, \dots, 50\}$. Interpolar la trayectoria continua de referencia sobre estos puntos discretos produce la serie temporal libre de ruido, denominada $I^{\text{clean}}(t_k)$.


In [ ]:
def muestrear_dias_enteros(
    t_ref: np.ndarray,
    I_ref: np.ndarray,
    t_dias: np.ndarray = None
) -> tuple[np.ndarray, np.ndarray]:
    """
    Submuestrea la serie continua de infectados sobre instantes de tiempo discretos.
    """
    # TU CODIGO AQUI
    ...


In [ ]:
# Pruebas de validacion para muestrear_dias_enteros
t_dias, I_clean = muestrear_dias_enteros(t_ref, I_ref)
assert len(t_dias) == 51, f"Se esperaban 51 puntos diarios (0 a 50), obtenidos {len(t_dias)}"
assert np.isclose(t_dias[0], 0.0) and np.isclose(t_dias[-1], 50.0), "Los extremos temporales no coinciden."
assert np.all(I_clean >= 0.0) and np.all(I_clean <= 1.0), "Valores fuera del rango fisico [0, 1]."
print("Validacion de muestrear_dias_enteros: EXITOSA.")


## 4. Proceso de Medicion: Ruido Gaussiano y Proyeccion Fisica

Para modelar el error observacional aditivo, perturbamos las mediciones con variables aleatorias normales independientes e identicamente distribuidas:

$$\varepsilon_k \sim \mathcal{N}(0, \sigma^2), \quad \sigma = 0.02$$

Dado que $I(t)$ representa la proporcion de la poblacion infectada, una observacion negativa es fisicamente inadmisible. Imponemos la proyeccion no lineal sobre el conjunto de estados validos mediante la operacion de truncamiento inferior:

$$I^{\text{obs}}_k = \max(0, I^{\text{clean}}(t_k) + \varepsilon_k)$$

Para garantizar reproducibilidad cientifica estricta, fijamos la semilla del generador pseudoaleatorio.


In [ ]:
def agregar_ruido_gaussiano(
    I_clean: np.ndarray,
    sigma: float = 0.02,
    seed: int = 42
) -> np.ndarray:
    """
    Agrega ruido aditivo Gaussiano N(0, sigma^2) con proyeccion a no-negatividad.
    """
    # TU CODIGO AQUI
    ...


In [ ]:
# Pruebas de validacion para agregar_ruido_gaussiano
I_obs_test = agregar_ruido_gaussiano(I_clean, sigma=0.02, seed=42)
assert I_obs_test.shape == I_clean.shape, "La forma del arreglo no coincide."
assert np.all(I_obs_test >= 0.0), "Violacion de no-negatividad fisica."
# Verificar determinismo mediante semilla fija
I_obs_test2 = agregar_ruido_gaussiano(I_clean, sigma=0.02, seed=42)
assert np.array_equal(I_obs_test, I_obs_test2), "El generador debe ser determinista con semilla fija."
print("Validacion de agregar_ruido_gaussiano: EXITOSA.")


## 5. Analisis Estadistico de Residuos Observacionales

Definimos el residuo observacional en cada instante muestral como la desviacion entre el valor observado y el valor real subyacente:

$$e_k = I^{\text{obs}}_k - I^{\text{clean}}(t_k)$$

Propiedades estadisticas a verificar:
1. **Media muestral**: $\bar{e} = \frac{1}{M} \sum_{k=1}^M e_k \approx 0$, lo que confirma la ausencia de sesgo sistematico en el proceso de captura.
2. **Desviacion estandar muestral**: $s_e = \sqrt{\frac{1}{M-1} \sum_{k=1}^M (e_k - \bar{e})^2} \approx \sigma = 0.02$.
3. **Banda de confianza $\pm 2\sigma$**: Para una distribucion Gaussiana, aproximadamente el $95.4\%$ de las observaciones deben residir dentro del intervalo $[-\text{2}\sigma, +\text{2}\sigma]$.


In [ ]:
def analizar_residuos(
    I_clean: np.ndarray,
    I_obs: np.ndarray,
    sigma: float = 0.02
) -> dict[str, Any]:
    """
    Calcula los estadisticos descriptivos de los residuos observacionales.
    """
    # TU CODIGO AQUI
    ...


In [ ]:
# Pruebas de validacion para analizar_residuos
stats = analizar_residuos(I_clean, I_obs_test, sigma=0.02)
assert isinstance(stats, dict), "La funcion debe retornar un diccionario."
assert "media" in stats and "desviacion_std" in stats and "porcentaje_2sigma" in stats
assert abs(stats["media"]) < 0.01, f"Media residual excesiva: {stats['media']}"
assert 0.01 < stats["desviacion_std"] < 0.03, f"Desviacion residual anormal: {stats['desviacion_std']}"
print("Validacion de analizar_residuos: EXITOSA.")


## 6. Generacion Completa y Caracterizacion Grafica

Ejecutamos el flujo completo de generacion con la semilla $42$ y $\sigma = 0.02$. Construimos dos graficos cientificos:
1. **Curva de referencia continua frente a las observaciones discretas**, incluyendo la banda de dispersion $\pm 2\sigma$.
2. **Serie temporal de los residuos**, contrastada contra las bandas de control $\pm 2\sigma$.


In [ ]:
sigma = 0.02
seed = 42

t_dias, I_clean = muestrear_dias_enteros(t_ref, I_ref)
I_obs = agregar_ruido_gaussiano(I_clean, sigma=sigma, seed=seed)
analisis = analizar_residuos(I_clean, I_obs, sigma=sigma)

print("=" * 65)
print("ANALISIS ESTADISTICO DE RESIDUOS OBSERVACIONALES")
print("=" * 65)
print(f"Numero de observaciones discretas: {len(t_dias)}")
print(f"Desviacion estandar nominal (sigma): {sigma:.4f}")
print(f"Media muestral de residuos:        {analisis['media']:+.5f}")
print(f"Desviacion estandar muestral (s):  {analisis['desviacion_std']:.5f}")
print(f"Observaciones en banda +- 2*sigma: {analisis['porcentaje_2sigma']:.1f}% (esperado: ~95.4%)")
print("=" * 65)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Panel 1: Curva de verdad de campo y observaciones ruidosas
ax1 = axes[0]
ax1.plot(t_ref, I_ref, 'k-', linewidth=2.0, label="Verdad de Campo I(t) [RK45]", zorder=2)
ax1.fill_between(
    t_dias,
    np.maximum(0.0, I_clean - 2 * sigma),
    I_clean + 2 * sigma,
    color="#9ecae1",
    alpha=0.45,
    label="Banda de Incertidumbre +- 2*sigma (95.4%)",
    zorder=1
)
ax1.scatter(
    t_dias, I_obs, color="#de2d26", s=28, facecolors="none", edgecolors="#de2d26",
    linewidths=1.5, label="Observaciones Clinicas Sinteticas", zorder=3
)
ax1.set_title("Serie Temporal: Dinamica Continua vs Observaciones")
ax1.set_xlabel("Tiempo (dias)")
ax1.set_ylabel("Proporcion de Infectados I(t)")
ax1.legend(frameon=True, loc="upper right")

# Panel 2: Serie temporal de residuos con bandas de control
ax2 = axes[1]
residuos = analisis["residuos"]
ax2.axhline(0.0, color="black", linestyle="-", linewidth=1.0)
ax2.axhline(+2 * sigma, color="#e6550d", linestyle="--", linewidth=1.5, label="+- 2*sigma (+- 0.04)")
ax2.axhline(-2 * sigma, color="#e6550d", linestyle="--", linewidth=1.5)
markerline, stemlines, baseline = ax2.stem(t_dias, residuos, linefmt="b-", markerfmt="bo", basefmt="k-")
plt.setp(stemlines, linewidth=1.2, color="#3182bd")
plt.setp(markerline, markersize=4, color="#3182bd")
plt.setp(baseline, linewidth=0.8, color="gray")
ax2.set_title("Residuos Observacionales: e_k = I_obs - I_clean")
ax2.set_xlabel("Tiempo (dias)")
ax2.set_ylabel("Residuo")
ax2.legend(frameon=True, loc="upper right")

plt.tight_layout()
plt.show()


## 7. Lectura Complementaria: Datos Discretos de Conteo

En este laboratorio hemos modelado el error mediante ruido Gaussiano continuo y homocedastico. Sin embargo, en entornos hospitalarios donde se registran conteos enteros de personas (por ejemplo, admisiones diarias), la variabilidad del proceso de notificacion es intrinsecamente discreta y heterocedastica: la varianza de los datos depende directamente de la magnitud media esperada de los casos.

Para los estudiantes interesados en profundizar en el modelado estadistico de conteo mediante procesos Poissonianos y verosimilitud de Poisson, se recomienda consultar el documento conceptual de apoyo:

[Lectura Complementaria: Modelado de Ruido de Conteo y Distribucion de Poisson](../../appendix/ruido_conteo_poisson.md)
